# 🏗️ Notebook 1: Google Search — Requirements & Architecture

## 🛠️ Setup

```bash
cd 06-system-designs/google-search
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## What we're designing

A web-scale search engine: take a text query, return the most relevant pages in <200ms.

### Functional requirements
- Crawl the web, follow links, fetch pages.
- **Index** pages by the words they contain.
- Given a query, return the top-K most relevant documents.
- Rank by relevance + quality + personalization.
- Suggestions (typeahead) — see the typeahead lab.

### Non-functional
- **Huge scale**: 50B+ pages, trillions of terms, hundreds of millions of queries/day.
- **Low latency** for queries (<200ms).
- Reasonable **freshness** (news within minutes, long-tail within weeks).


## Three pipelines

```
   ┌──────────┐   fetch  ┌──────────┐  parse  ┌──────────┐
   │ Crawler  │─────────▶│ Document │────────▶│ Indexer  │
   └──────────┘          │  Store   │         └──────────┘
        ▲                └──────────┘              │
        │ seed + discovered                        ▼
        │                                   ┌────────────┐
        │                                   │ Inverted   │
        │                                   │ Index      │
        │                                   └────────────┘
        │                                         ▲
        │                                         │ lookup
        │                                   ┌────────────┐
        └─────── Ranking/Quality signals ───│ Query Svc  │◀── user
                                            └────────────┘
```

Three *independent* pipelines:
1. **Crawling** — build a document corpus.
2. **Indexing** — preprocess docs into an inverted index.
3. **Serving** — answer queries against the index.


## Back-of-envelope

- 50B pages × 100 KB = **5 PB** of raw HTML.
- Inverted index (compressed): ~10–20% of raw → ~500 TB.
- Sharded across 1000s of machines; each shard holds a slice of the vocabulary or doc-id range.
- Query: hits every shard in parallel, partial results merged.


## Back-of-envelope - let the code do the math

Rough numbers matter when you design a system. Instead of guessing, let's *compute*
the sizing with simple Python - change the inputs and see the impact.


In [ ]:
# Capacity estimation for a web-scale search engine.
# All numbers are rough - the point is to practice thinking in orders of magnitude.

PAGES           = 50_000_000_000     # 50B pages crawled
AVG_PAGE_KB     = 100                # average raw HTML size
INDEX_RATIO     = 0.15               # compressed inverted index ~ 15% of raw
QPS             = 100_000            # queries per second at peak
AVG_DOC_BYTES_RETURNED = 500         # snippet + url + title in a result
RESULTS_PER_QUERY = 10

raw_pb    = PAGES * AVG_PAGE_KB * 1024 / (1024**5)
index_pb  = raw_pb * INDEX_RATIO
egress_gbps = QPS * RESULTS_PER_QUERY * AVG_DOC_BYTES_RETURNED * 8 / 1e9

print(f'Raw HTML corpus : {raw_pb:,.1f} PB')
print(f'Inverted index  : {index_pb*1024:,.0f} TB  (compressed)')
print(f'Query egress    : {egress_gbps:,.1f} Gbps at peak QPS')

# Shard sizing: if one box stores 500 GB of index, we need:
shard_bytes = 500 * 1024**3
total_index_bytes = index_pb * 1024**5
print(f'Shards needed   : {total_index_bytes/shard_bytes:,.0f} '
      f'(at {shard_bytes/1024**3:.0f} GB/shard)')


## Tiny end-to-end search - the whole system in 20 lines

Before zooming into each pipeline, here is the **entire** crawl -> index -> query flow on 4 pages.
Every later notebook replaces one of these steps with a **better** version.


In [ ]:
# Minimum-viable search: crawl (mock), index, query.
from collections import defaultdict
import re

# 1) 'Crawl' - normally HTTP fetches; here we pretend.
CORPUS = {
    'https://a.com/py':    'Python is a popular programming language',
    'https://a.com/flask': 'Flask is a lightweight Python web framework',
    'https://b.com/django':'Django is a batteries-included Python web framework',
    'https://b.com/rust':  'Rust is a fast systems programming language',
}

# 2) Index - tokenize + build term -> set(doc_id)
def tok(t): return re.findall(r'[a-z]+', t.lower())
index = defaultdict(set)
docs = list(CORPUS.items())                   # doc_id = position in list
for doc_id, (_, text) in enumerate(docs):
    for w in tok(text):
        index[w].add(doc_id)

# 3) Query - intersect postings, return URLs
def search(q):
    terms = tok(q)
    if not terms: return []
    ids = set.intersection(*(index.get(t, set()) for t in terms))
    return [docs[i][0] for i in ids]

print('python web framework ->', search('python web framework'))
print('programming language ->', search('programming language'))
print('javascript           ->', search('javascript'))  # empty result


Everything from here is about making **each of those three lines of code** survive the
real internet: billions of pages, noisy text, hostile servers, and sub-second queries.
